In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# ========================================================================================
# 1. HANDLE CLASS IMBALANCE
# ========================================================================================

def handle_class_imbalance(X, y, method='class_weight'):
    """
    Handle class imbalance using different strategies
    """
    if method == 'class_weight':
        # Calculate class weights
        classes = np.unique(y)
        class_weights = compute_class_weight('balanced', classes=classes, y=y)
        weight_dict = dict(zip(classes, class_weights))
        return X, y, weight_dict
    
    elif method == 'smote':
        # SMOTE oversampling
        smote = SMOTE(random_state=42, k_neighbors=5)
        X_resampled, y_resampled = smote.fit_resample(X, y)
        return X_resampled, y_resampled, None
    
    elif method == 'undersample':
        # Random undersampling
        undersampler = RandomUnderSampler(random_state=42)
        X_resampled, y_resampled = undersampler.fit_resample(X, y)
        return X_resampled, y_resampled, None
    
    elif method == 'hybrid':
        # Combine oversampling and undersampling
        over = SMOTE(sampling_strategy=0.5, random_state=42)  # Increase minority class
        under = RandomUnderSampler(sampling_strategy=0.8, random_state=42)  # Reduce majority class
        
        steps = [('over', over), ('under', under)]
        pipeline = ImbPipeline(steps=steps)
        X_resampled, y_resampled = pipeline.fit_resample(X, y)
        return X_resampled, y_resampled, None

# ========================================================================================
# 2. PROFIT-OPTIMIZED THRESHOLD
# ========================================================================================

def profit_optimized_threshold(y_true, y_prob, loan_amounts, profit_per_good=2324.67, loss_per_bad=8271.76):
    """
    Find threshold that maximizes profit instead of just accuracy
    """
    thresholds = np.arange(0.1, 0.9, 0.01)
    profits = []
    
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        
        # Calculate profit for each prediction
        total_profit = 0
        for i in range(len(y_true)):
            if y_pred[i] == 1:  # Predicted default - reject loan
                total_profit += 0  # No profit/loss
            else:  # Predicted good - approve loan
                if y_true[i] == 0:  # Actually good
                    total_profit += profit_per_good
                else:  # Actually bad
                    total_profit -= loss_per_bad
        
        profits.append(total_profit)
    
    optimal_idx = np.argmax(profits)
    optimal_threshold = thresholds[optimal_idx]
    max_profit = profits[optimal_idx]
    
    return optimal_threshold, max_profit, thresholds, profits

# ========================================================================================
# 3. IMPROVED XGBOOST MODEL
# ========================================================================================

def train_improved_xgb(X_train, y_train, X_test, y_test, method='class_weight'):
    """
    Train improved XGBoost model with better parameters
    """
    print(f"Training XGBoost with {method} method...")
    
    # Handle class imbalance
    if method == 'class_weight':
        X_train_balanced, y_train_balanced, class_weights = handle_class_imbalance(X_train, y_train, method)
        sample_weight = None
    else:
        X_train_balanced, y_train_balanced, _ = handle_class_imbalance(X_train, y_train, method)
        class_weights = None
        sample_weight = None
    
    # Improved XGBoost parameters
    xgb_params = {
        'n_estimators': 500,
        'max_depth': 6,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'n_jobs': -1,
        'early_stopping_rounds': 50,
        'eval_metric': 'auc'
    }
    
    # Add class weights if using that method
    if class_weights:
        scale_pos_weight = class_weights[0] / class_weights[1]
        xgb_params['scale_pos_weight'] = scale_pos_weight
    
    # Train model
    model = XGBClassifier(**xgb_params)
    
    # Fit with early stopping
    model.fit(
        X_train_balanced, y_train_balanced,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    # Predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    auc = roc_auc_score(y_test, y_prob)
    
    print(f"XGBoost ({method}) ROC-AUC: {auc:.4f}")
    print(classification_report(y_test, y_pred))
    
    return model, y_prob, auc

# ========================================================================================
# 4. FEATURE ENGINEERING IMPROVEMENTS
# ========================================================================================

def create_advanced_features(df):
    """
    Create advanced features for better model performance
    """
    df_features = df.copy()
    
    # Debt-to-income ratio
    if 'annual_inc' in df.columns and 'loan_amnt' in df.columns:
        df_features['debt_to_income'] = df_features['loan_amnt'] / (df_features['annual_inc'] + 1)
    
    # Credit utilization
    if 'revol_bal' in df.columns and 'revol_util' in df.columns:
        df_features['credit_utilization'] = df_features['revol_util'] / 100
    
    # Employment length features
    if 'emp_length' in df.columns:
        df_features['emp_length'] = df_features['emp_length'].astype(str)
        df_features['emp_length_numeric'] = df_features['emp_length'].str.extract(r'(\d+)').astype(float)
        df_features['emp_length_numeric'] = df_features['emp_length_numeric'].fillna(0)

    
    # Loan amount to income ratio
    if 'annual_inc' in df.columns and 'loan_amnt' in df.columns:
        df_features['loan_to_income_ratio'] = df_features['loan_amnt'] / (df_features['annual_inc'] + 1)
    
    # Interest rate categories
    if 'int_rate' in df.columns:
        df_features['int_rate_category'] = pd.cut(df_features['int_rate'], 
                                                 bins=[0, 10, 15, 20, 30], 
                                                 labels=['Low', 'Medium', 'High', 'Very High'])
    
    # Credit score features (if available)
    if 'fico_range_low' in df.columns and 'fico_range_high' in df.columns:
        df_features['fico_avg'] = (df_features['fico_range_low'] + df_features['fico_range_high']) / 2
        df_features['fico_range'] = df_features['fico_range_high'] - df_features['fico_range_low']
    
    return df_features

# ========================================================================================
# 5. HYPERPARAMETER TUNING
# ========================================================================================

def hyperparameter_tuning(X_train, y_train, cv_folds=3):
    """
    Perform hyperparameter tuning for XGBoost
    """
    param_grid = {
        'n_estimators': [300, 500, 700],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1, 0.2],
        'subsample': [0.8, 0.9],
        'colsample_bytree': [0.8, 0.9],
        'reg_alpha': [0, 0.1, 0.5],
        'reg_lambda': [1, 1.5, 2]
    }
    
    # Calculate class weights
    classes = np.unique(y_train)
    class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
    scale_pos_weight = class_weights[0] / class_weights[1]
    
    # Base model
    xgb_base = XGBClassifier(
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight
    )
    
    # Grid search
    grid_search = GridSearchCV(
        xgb_base,
        param_grid,
        cv=StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42),
        scoring='roc_auc',
        n_jobs=-1,
        verbose=1
    )
    
    print("Starting hyperparameter tuning...")
    grid_search.fit(X_train, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_

# ========================================================================================
# 6. SEGMENT-SPECIFIC MODELS
# ========================================================================================

def train_segment_specific_models(df, target_col='loan_status'):
    """
    Train different models for different risk segments
    """
    models = {}
    
    # Create risk segments based on loan grade or amount
    if 'grade' in df.columns:
        df['risk_segment'] = df['grade'].map({
            'A': 'Low', 'B': 'Low',
            'C': 'Medium', 'D': 'Medium',
            'E': 'High', 'F': 'High', 'G': 'High'
        })
    else:
        # Create segments based on loan amount
        df['risk_segment'] = pd.cut(df['loan_amnt'], 
                                   bins=3, 
                                   labels=['Low', 'Medium', 'High'])
    
    # Train model for each segment
    for segment in df['risk_segment'].unique():
        if pd.isna(segment):
            continue
            
        print(f"\nTraining model for {segment} risk segment...")
        
        segment_data = df[df['risk_segment'] == segment]
        
        # Prepare features (assuming target is binary)
        X = segment_data.drop([target_col, 'risk_segment'], axis=1)
        y = segment_data[target_col]
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Train model
        model, y_prob, auc = train_improved_xgb(X_train, y_train, X_test, y_test)
        
        models[segment] = {
            'model': model,
            'auc': auc,
            'test_data': (X_test, y_test, y_prob)
        }
    
    return models

# ========================================================================================
# 7. MODEL COMPARISON AND SELECTION
# ========================================================================================

def compare_models(X_train, y_train, X_test, y_test):
    """
    Compare different approaches and select the best one
    """
    results = {}
    
    methods = ['class_weight']
    
    for method in methods:
        print(f"\n{'='*50}")
        print(f"Testing {method.upper()} method")
        print(f"{'='*50}")
        
        model, y_prob, auc = train_improved_xgb(X_train, y_train, X_test, y_test, method)
        
        results[method] = {
            'model': model,
            'auc': auc,
            'y_prob': y_prob
        }
    
    # Find best method
    best_method = max(results.keys(), key=lambda x: results[x]['auc'])
    print(f"\nBest method: {best_method} with AUC: {results[best_method]['auc']:.4f}")
    
    return results, best_method

# ========================================================================================
# 8. MAIN EXECUTION FUNCTION
# ========================================================================================

def improve_credit_risk_model(X, y):
    """
    Main function to improve your credit risk model (expects preprocessed X and y)
    """
    print("Starting Credit Risk Model Improvement Process...")
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # 1. Compare different class imbalance methods
    print("\n1. Comparing different class imbalance handling methods...")
    results, best_method = compare_models(X_train, y_train, X_test, y_test)
    
    # 2. Hyperparameter tuning on the best method
    print(f"\n2. Hyperparameter tuning for {best_method} method...")
    best_model = hyperparameter_tuning(X_train, y_train)
    
    # 3. Profit-optimized threshold
    print("\n3. Finding profit-optimized threshold...")
    y_prob_final = best_model.predict_proba(X_test)[:, 1]
    
    # You can replace loan_amounts with actual values per sample if available
    optimal_threshold, max_profit, thresholds, profits = profit_optimized_threshold(
        y_test, y_prob_final,
        loan_amounts=np.ones(len(y_test)) * 15000,
        profit_per_good=2324.67,
        loss_per_bad=8271.76
    )
    
    print(f"Optimal threshold: {optimal_threshold:.3f}")
    print(f"Maximum profit: ${max_profit:,.2f}")
    
    # 4. Final evaluation
    print("\n4. Final model evaluation...")
    y_pred_optimal = (y_prob_final >= optimal_threshold).astype(int)
    final_auc = roc_auc_score(y_test, y_prob_final)
    
    print(f"Final AUC: {final_auc:.4f}")
    print(classification_report(y_test, y_pred_optimal))
    
    return {
        'best_model': best_model,
        'optimal_threshold': optimal_threshold,
        'final_auc': final_auc,
        'feature_names': X.columns.tolist()
    }


# ========================================================================================
# USAGE EXAMPLE
# ========================================================================================
"""
# Load your data
df = pd.read_csv('your_lending_data.csv')

# Improve your model
results = improve_credit_risk_model(df, target_col='loan_status')

# Use the improved model
best_model = results['best_model']
optimal_threshold = results['optimal_threshold']

# Make predictions
new_predictions = best_model.predict_proba(new_data)[:, 1]
final_decisions = (new_predictions >= optimal_threshold).astype(int)
"""

print("Credit Risk Model Improvement Framework Ready!")
print("\nKey Improvements:")
print("1. ✅ Handle class imbalance (4 different methods)")
print("2. ✅ Profit-optimized thresholds")
print("3. ✅ Advanced feature engineering")
print("4. ✅ Hyperparameter tuning")
print("5. ✅ Segment-specific models")
print("6. ✅ Model comparison framework")
print("\nYour current AUC of 0.77 should improve to 0.80+ with these techniques!")

Credit Risk Model Improvement Framework Ready!

Key Improvements:
1. ✅ Handle class imbalance (4 different methods)
2. ✅ Profit-optimized thresholds
3. ✅ Advanced feature engineering
4. ✅ Hyperparameter tuning
5. ✅ Segment-specific models
6. ✅ Model comparison framework

Your current AUC of 0.77 should improve to 0.80+ with these techniques!


In [14]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import precision_recall_curve, confusion_matrix, fbeta_score
# from sklearn.metrics import classification_report, roc_curve, auc
# import warnings
# warnings.filterwarnings('ignore')

# class CreditRiskOptimizer:
#     def __init__(self, y_true, y_pred_proba, loan_amount=None, segments=None):
#         """
#         Initialize the credit risk optimizer
        
#         Parameters:
#         y_true: actual labels (0=no default, 1=default)
#         y_pred_proba: predicted probabilities for default class
#         loan_amount: loan amounts for each sample (optional)
#         segments: customer segments for each sample (optional)
#         """
#         self.y_true = y_true
#         self.y_pred_proba = y_pred_proba
#         self.loan_amount = loan_amount if loan_amount is not None else np.ones(len(y_true))
#         self.segments = segments
        
#         # Business parameters (you can adjust these)
#         self.avg_loss_per_default = 50000  # Average loss when loan defaults
#         self.avg_profit_per_loan = 5000    # Average profit from good loan
#         self.interest_rate = 0.12          # Annual interest rate
        
#     def calculate_business_metrics(self, threshold):
#         """Calculate business impact for a given threshold"""
#         y_pred = (self.y_pred_proba >= threshold).astype(int)
        
#         # Confusion matrix
#         tn, fp, fn, tp = confusion_matrix(self.y_true, y_pred).ravel()
        
#         # Business calculations
#         # TP: Correctly identified defaults (saved losses)
#         # FP: Rejected good customers (lost profits)
#         # FN: Missed defaults (actual losses)
#         # TN: Approved good customers (gained profits)
        
#         saved_losses = tp * self.avg_loss_per_default
#         lost_profits = fp * self.avg_profit_per_loan
#         actual_losses = fn * self.avg_loss_per_default
#         gained_profits = tn * self.avg_profit_per_loan
        
#         net_benefit = gained_profits + saved_losses - actual_losses - lost_profits
        
#         return {
#             'threshold': threshold,
#             'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
#             'precision': tp / (tp + fp) if (tp + fp) > 0 else 0,
#             'recall': tp / (tp + fn) if (tp + fn) > 0 else 0,
#             'saved_losses': saved_losses,
#             'lost_profits': lost_profits,
#             'actual_losses': actual_losses,
#             'gained_profits': gained_profits,
#             'net_benefit': net_benefit,
#             'approval_rate': (tn + fn) / len(self.y_true)
#         }
    
#     def optimize_threshold_expected_value(self, thresholds=None):
#         """1. Calculate expected value for different thresholds"""
#         if thresholds is None:
#             thresholds = np.arange(0.1, 0.9, 0.05)
        
#         results = []
#         for threshold in thresholds:
#             results.append(self.calculate_business_metrics(threshold))
        
#         df_results = pd.DataFrame(results)
        
#         # Find optimal threshold
#         optimal_idx = df_results['net_benefit'].idxmax()
#         optimal_threshold = df_results.loc[optimal_idx, 'threshold']
        
#         print("=== EXPECTED VALUE ANALYSIS ===")
#         print(f"Optimal Threshold: {optimal_threshold:.3f}")
#         print(f"Maximum Net Benefit: ${df_results.loc[optimal_idx, 'net_benefit']:,.0f}")
#         print(f"Precision: {df_results.loc[optimal_idx, 'precision']:.3f}")
#         print(f"Recall: {df_results.loc[optimal_idx, 'recall']:.3f}")
#         print(f"Approval Rate: {df_results.loc[optimal_idx, 'approval_rate']:.3f}")
        
#         return df_results, optimal_threshold
    
#     def analyze_fbeta_scores(self, beta_values=[0.5, 1.0, 1.5, 2.0]):
#         """2. Use F-beta score analysis"""
#         thresholds = np.arange(0.1, 0.9, 0.05)
        
#         results = {}
#         print("\n=== F-BETA SCORE ANALYSIS ===")
        
#         for beta in beta_values:
#             best_threshold = 0
#             best_fbeta = 0
            
#             for threshold in thresholds:
#                 y_pred = (self.y_pred_proba >= threshold).astype(int)
#                 fbeta = fbeta_score(self.y_true, y_pred, beta=beta)
                
#                 if fbeta > best_fbeta:
#                     best_fbeta = fbeta
#                     best_threshold = threshold
            
#             results[beta] = {'threshold': best_threshold, 'fbeta': best_fbeta}
            
#             # Get metrics for best threshold
#             metrics = self.calculate_business_metrics(best_threshold)
            
#             print(f"Beta = {beta} (recall weight = {beta}, precision weight = 1)")
#             print(f"  Best Threshold: {best_threshold:.3f}")
#             print(f"  F-beta Score: {best_fbeta:.3f}")
#             print(f"  Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}")
#             print(f"  Net Benefit: ${metrics['net_benefit']:,.0f}")
#             print()
        
#         return results
    
#     def profit_based_optimization(self):
#         """3. Profit-based metrics optimization"""
#         print("=== PROFIT-BASED OPTIMIZATION ===")
        
#         # Try different business scenarios
#         scenarios = [
#             {"name": "Conservative (High Loss Aversion)", "loss": 100000, "profit": 3000},
#             {"name": "Balanced", "loss": 50000, "profit": 5000},
#             {"name": "Aggressive (Growth Focus)", "loss": 30000, "profit": 8000}
#         ]
        
#         thresholds = np.arange(0.1, 0.9, 0.05)
#         scenario_results = {}
        
#         for scenario in scenarios:
#             # Temporarily update business parameters
#             original_loss = self.avg_loss_per_default
#             original_profit = self.avg_profit_per_loan
            
#             self.avg_loss_per_default = scenario["loss"]
#             self.avg_profit_per_loan = scenario["profit"]
            
#             best_threshold = 0
#             best_net_benefit = float('-inf')
            
#             for threshold in thresholds:
#                 metrics = self.calculate_business_metrics(threshold)
#                 if metrics['net_benefit'] > best_net_benefit:
#                     best_net_benefit = metrics['net_benefit']
#                     best_threshold = threshold
            
#             scenario_results[scenario["name"]] = {
#                 'threshold': best_threshold,
#                 'net_benefit': best_net_benefit,
#                 'metrics': self.calculate_business_metrics(best_threshold)
#             }
            
#             print(f"{scenario['name']}:")
#             print(f"  Loss per default: ${scenario['loss']:,}")
#             print(f"  Profit per loan: ${scenario['profit']:,}")
#             print(f"  Optimal threshold: {best_threshold:.3f}")
#             print(f"  Net benefit: ${best_net_benefit:,.0f}")
#             print(f"  Precision: {scenario_results[scenario['name']]['metrics']['precision']:.3f}")
#             print(f"  Recall: {scenario_results[scenario['name']]['metrics']['recall']:.3f}")
#             print()
            
#             # Restore original parameters
#             self.avg_loss_per_default = original_loss
#             self.avg_profit_per_loan = original_profit
        
#         return scenario_results
    
#     def segment_analysis(self, segment_names=None):
#         """4. Segment-based threshold optimization"""
#         if self.segments is None:
#             print("=== SEGMENT ANALYSIS ===")
#             print("No segments provided. Creating example segments based on predicted probability quartiles.")
            
#             # Create example segments based on risk quartiles
#             quartiles = np.percentile(self.y_pred_proba, [25, 50, 75])
#             segments = np.digitize(self.y_pred_proba, quartiles)
#             segment_names = ['Low Risk', 'Medium-Low Risk', 'Medium-High Risk', 'High Risk']
#             self.segments = segments
        
#         if segment_names is None:
#             segment_names = [f'Segment {i}' for i in range(len(np.unique(self.segments)))]
        
#         print("=== SEGMENT ANALYSIS ===")
        
#         segment_results = {}
#         thresholds = np.arange(0.1, 0.9, 0.05)
        
#         for segment_id in np.unique(self.segments):
#             segment_mask = self.segments == segment_id
#             segment_name = segment_names[segment_id] if segment_id < len(segment_names) else f'Segment {segment_id}'
            
#             if np.sum(segment_mask) == 0:
#                 continue
            
#             # Get segment data
#             y_true_seg = self.y_true[segment_mask]
#             y_pred_proba_seg = self.y_pred_proba[segment_mask]
            
#             # Find optimal threshold for this segment
#             best_threshold = 0
#             best_net_benefit = float('-inf')
            
#             for threshold in thresholds:
#                 y_pred_seg = (y_pred_proba_seg >= threshold).astype(int)
                
#                 if len(np.unique(y_pred_seg)) < 2:  # Skip if all predictions are the same
#                     continue
                
#                 try:
#                     tn, fp, fn, tp = confusion_matrix(y_true_seg, y_pred_seg).ravel()
                    
#                     # Calculate net benefit for segment
#                     saved_losses = tp * self.avg_loss_per_default
#                     lost_profits = fp * self.avg_profit_per_loan
#                     actual_losses = fn * self.avg_loss_per_default
#                     gained_profits = tn * self.avg_profit_per_loan
#                     net_benefit = gained_profits + saved_losses - actual_losses - lost_profits
                    
#                     if net_benefit > best_net_benefit:
#                         best_net_benefit = net_benefit
#                         best_threshold = threshold
#                 except:
#                     continue
            
#             # Calculate final metrics for best threshold
#             y_pred_seg = (y_pred_proba_seg >= best_threshold).astype(int)
#             tn, fp, fn, tp = confusion_matrix(y_true_seg, y_pred_seg).ravel()
            
#             precision = tp / (tp + fp) if (tp + fp) > 0 else 0
#             recall = tp / (tp + fn) if (tp + fn) > 0 else 0
#             default_rate = np.mean(y_true_seg)
            
#             segment_results[segment_name] = {
#                 'threshold': best_threshold,
#                 'net_benefit': best_net_benefit,
#                 'precision': precision,
#                 'recall': recall,
#                 'default_rate': default_rate,
#                 'size': np.sum(segment_mask)
#             }
            
#             print(f"{segment_name} (n={np.sum(segment_mask):,}):")
#             print(f"  Default rate: {default_rate:.3f}")
#             print(f"  Optimal threshold: {best_threshold:.3f}")
#             print(f"  Precision: {precision:.3f}, Recall: {recall:.3f}")
#             print(f"  Net benefit: ${best_net_benefit:,.0f}")
#             print()
        
#         return segment_results
    
#     def plot_optimization_results(self, df_results):
#         """Create visualization plots"""
#         fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
#         # 1. Net Benefit vs Threshold
#         axes[0, 0].plot(df_results['threshold'], df_results['net_benefit'] / 1000000, 'b-', linewidth=2)
#         axes[0, 0].set_xlabel('Threshold')
#         axes[0, 0].set_ylabel('Net Benefit ($ Millions)')
#         axes[0, 0].set_title('Net Benefit vs Threshold')
#         axes[0, 0].grid(True, alpha=0.3)
        
#         # 2. Precision-Recall Tradeoff
#         axes[0, 1].plot(df_results['recall'], df_results['precision'], 'r-', linewidth=2)
#         axes[0, 1].set_xlabel('Recall')
#         axes[0, 1].set_ylabel('Precision')
#         axes[0, 1].set_title('Precision-Recall Tradeoff')
#         axes[0, 1].grid(True, alpha=0.3)
        
#         # 3. Approval Rate vs Threshold
#         axes[1, 0].plot(df_results['threshold'], df_results['approval_rate'], 'g-', linewidth=2)
#         axes[1, 0].set_xlabel('Threshold')
#         axes[1, 0].set_ylabel('Approval Rate')
#         axes[1, 0].set_title('Approval Rate vs Threshold')
#         axes[1, 0].grid(True, alpha=0.3)
        
#         # 4. Components of Net Benefit
#         axes[1, 1].plot(df_results['threshold'], df_results['gained_profits'] / 1000000, 
#                        label='Gained Profits', linewidth=2)
#         axes[1, 1].plot(df_results['threshold'], df_results['saved_losses'] / 1000000, 
#                        label='Saved Losses', linewidth=2)
#         axes[1, 1].plot(df_results['threshold'], -df_results['actual_losses'] / 1000000, 
#                        label='Actual Losses', linewidth=2)
#         axes[1, 1].plot(df_results['threshold'], -df_results['lost_profits'] / 1000000, 
#                        label='Lost Profits', linewidth=2)
#         axes[1, 1].set_xlabel('Threshold')
#         axes[1, 1].set_ylabel('Amount ($ Millions)')
#         axes[1, 1].set_title('Components of Net Benefit')
#         axes[1, 1].legend()
#         axes[1, 1].grid(True, alpha=0.3)
        
#         plt.tight_layout()
#         plt.show()
    
#     def run_complete_analysis(self):
#         """Run all four optimization approaches"""
#         print("CREDIT RISK THRESHOLD OPTIMIZATION")
#         print("=" * 50)
        
#         # 1. Expected Value Analysis
#         df_results, optimal_threshold = self.optimize_threshold_expected_value()
        
#         # 2. F-beta Score Analysis
#         fbeta_results = self.analyze_fbeta_scores()
        
#         # 3. Profit-based Optimization
#         profit_results = self.profit_based_optimization()
        
#         # 4. Segment Analysis
#         segment_results = self.segment_analysis()
        
#         # Plot results
#         self.plot_optimization_results(df_results)
        
#         return {
#             'expected_value': df_results,
#             'fbeta_results': fbeta_results,
#             'profit_scenarios': profit_results,
#             'segment_results': segment_results,
#             'optimal_threshold': optimal_threshold
#         }

# # Example usage with your data:
# """
# # Assuming you have:
# # y_true: actual labels from your test set
# # y_pred_proba: predicted probabilities from your XGBoost model
# # 
# # Example:
# # y_true = your_test_labels
# # y_pred_proba = your_model.predict_proba(X_test)[:, 1]  # probabilities for class 1
# # 
# # optimizer = CreditRiskOptimizer(y_true, y_pred_proba)
# # results = optimizer.run_complete_analysis()
# """

# print("Credit Risk Optimizer created!")
# print("\nTo use this with your XGBoost model:")
# print("1. Get predicted probabilities: y_pred_proba = model.predict_proba(X_test)[:, 1]")
# print("2. Create optimizer: optimizer = CreditRiskOptimizer(y_true, y_pred_proba)")
# print("3. Run analysis: results = optimizer.run_complete_analysis()")
# print("\nYou can also customize business parameters:")
# print("- optimizer.avg_loss_per_default = 50000")
# print("- optimizer.avg_profit_per_loan = 5000")

In [16]:
import joblib

In [17]:
X_processed = joblib.load("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/X_processed_1.pkl")

y = joblib.load("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/y_1.pkl")

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42
)


In [19]:
# results = improve_credit_risk_model(X_processed, y)

In [36]:
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.06,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=3,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train, y_train)
y_xgb_pred = xgb.predict(X_test)
y_xgb_prob = xgb.predict_proba(X_test)[:, 1]

print("XGBoost ROC-AUC:", roc_auc_score(y_test, y_xgb_prob))
print(classification_report(y_test, y_xgb_pred))

XGBoost ROC-AUC: 0.7712885079511889
              precision    recall  f1-score   support

           0       0.88      0.83      0.86    299557
           1       0.44      0.54      0.48     72510

    accuracy                           0.77    372067
   macro avg       0.66      0.68      0.67    372067
weighted avg       0.79      0.77      0.78    372067



In [35]:
# from xgboost import XGBClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import roc_auc_score, classification_report
# import pandas as pd

# def train_segment_specific_models(X_processed, y, df_raw):
#     """
#     Train XGBoost models separately for each risk segment using user-defined XGB setup.
#     """

#     # Convert X and y to DataFrame/Series for indexing
#     if isinstance(X_processed, np.ndarray):
#         X_processed = pd.DataFrame(X_processed)
#     if isinstance(y, np.ndarray):
#         y = pd.Series(y)

#     X_processed = X_processed.reset_index(drop=True)
#     df_raw = df_raw.reset_index(drop=True)
#     y = y.reset_index(drop=True)

#     # Step 1: Create risk segments based on 'grade' (if available)
#     if 'grade' in df_raw.columns:
#         df_raw['risk_segment'] = df_raw['grade'].map({
#             'A': 'Low', 'B': 'Low',
#             'C': 'Medium', 'D': 'Medium',
#             'E': 'High', 'F': 'High', 'G': 'High'
#         })
#     else:
#         raise ValueError("Column 'grade' is required in df_raw for segmentation.")

#     # Step 2: Append segment info to processed features
#     X_segmented = X_processed.copy()
#     X_segmented['risk_segment'] = df_raw['risk_segment']

#     models = {}

#     for segment in X_segmented['risk_segment'].dropna().unique():
#         print(f"\n🔹 Training model for '{segment}' risk segment...")

#         segment_mask = X_segmented['risk_segment'] == segment
#         X_seg = X_segmented[segment_mask].drop(columns=['risk_segment'])
#         y_seg = y[segment_mask]

#         X_train, X_test, y_train, y_test = train_test_split(
#             X_seg, y_seg, test_size=0.2, random_state=42, stratify=y_seg
#         )

#         # Step 3: Compute scale_pos_weight
#         scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

#         # Step 4: Train XGBoost using your preferred setup
#         xgb = XGBClassifier(
#             n_estimators=400,
#             max_depth=6,
#             learning_rate=0.06,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             eval_metric='logloss',
#             scale_pos_weight=scale_pos_weight,
#             use_label_encoder=False,
#             random_state=42,
#             n_jobs=-1
#         )

#         xgb.fit(X_train, y_train)

#         # Step 5: Evaluate
#         y_prob = xgb.predict_proba(X_test)[:, 1]
#         y_pred = (y_prob >= 0.5).astype(int)
#         auc = roc_auc_score(y_test, y_prob)

#         print(f"AUC for '{segment}': {auc:.4f}")
#         print(classification_report(y_test, y_pred))

#         models[segment] = {
#             'model': xgb,
#             'auc': auc,
#             'test_data': (X_test, y_test, y_prob)
#         }

#     return models


In [23]:
df1 = pd.read_csv("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/clean_credit_impute.csv")

In [34]:
# segment_models = train_segment_specific_models(X_processed, y, df1)

In [43]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Step 1: Predict probabilities
y_proba = xgb.predict_proba(X_test)[:, 1]

# Step 2: Apply fixed threshold
threshold = 0.5
y_pred = (y_proba >= threshold).astype(int)

# Step 3: Evaluate
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))


Confusion Matrix:
[[249123  50434]
 [ 33548  38962]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.83      0.86    299557
           1       0.44      0.54      0.48     72510

    accuracy                           0.77    372067
   macro avg       0.66      0.68      0.67    372067
weighted avg       0.79      0.77      0.78    372067

ROC-AUC Score: 0.7712885079511889
